In [1]:
from google.colab import files

uploaded = files.upload()

Saving sales.csv to sales.csv


In [2]:
import os

print(os.listdir())

['.config', 'sales.csv', 'sample_data']


In [3]:
import os
import shutil

# Create required folders
os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

# Copy the original file WITHOUT modifying it
shutil.copy(
    "sales.csv",
    "data_lake/bronze/sales.csv"
)

print("Data Lake structure created successfully.")

for root, dirs, files in os.walk("data_lake"):
    level = root.replace("data_lake", "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}    {file}")

Data Lake structure created successfully.
data_lake/
    bronze/
        sales.csv
    silver/
        preprocessed_sales/


In [4]:
!pip install pyspark

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("DataMiningAssignment") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.4


In [6]:
bronze_path = "data_lake/bronze/sales.csv"

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .option("mode", "PERMISSIVE") \
    .csv(bronze_path)

print("Bronze data loaded successfully.")

Bronze data loaded successfully.


In [7]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [8]:
df.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

In [9]:
df.limit(5).toPandas()

,order_id,customer_id,customer_name,product,category,quantity,unit_price,discount_percent,payment_method,city,state,order_date,order_status
0,100001,C0256,Keerthi Mishra,Saree,Clothing,2,6710,20,Credit Card,Mumbai,Maharashtra,2026-01-01,Delivered
1,100002,C0042,Arjun Singh,T-Shirt,Clothing,6,680,5,Debit Card,Chennai,Tamil Nadu,2025-08-03,Delivered
2,100003,C0348,Keerthi Rao,Hoodie,Clothing,6,3080,12,UPI,Pune,Maharashtra,2025-05-06,Shipped
3,100004,C0051,Tanvi Sharma,Running Shoes,Sports,6,2260,35,Net Banking,Lucknow,Uttar Pradesh,2026-07-31,Cancelled
4,100005,C0227,Aarav Kumar,Coffee 500g,Grocery,10,740,12,Debit Card,Vijayawada,Andhra Pradesh,2026-08-30,Delivered


In [10]:
before_count = df.count()

print("Total records before preprocessing:", before_count)

Total records before preprocessing: 1000


In [11]:
df.dtypes

[('order_id', 'int'),
 ('customer_id', 'string'),
 ('customer_name', 'string'),
 ('product', 'string'),
 ('category', 'string'),
 ('quantity', 'int'),
 ('unit_price', 'int'),
 ('discount_percent', 'int'),
 ('payment_method', 'string'),
 ('city', 'string'),
 ('state', 'string'),
 ('order_date', 'string'),
 ('order_status', 'string')]

In [13]:
duplicate_count = df.count() - df.dropDuplicates().count()

print("Number of duplicate rows:", duplicate_count)

df.groupBy(df.columns) \
  .count() \
  .filter(F.col("count") > 1) \
  .show(truncate=False)

Number of duplicate rows: 10
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|order_id|customer_id|customer_name   |product          |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|count|
+--------+-----------+----------------+-----------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+-----+
|100692  |C0141      |Varun Agarwal   |Electric Kettle  |Home Appliances|4       |920       |0               |Net Banking     |Delhi     |Delhi         |2026-08-18|Shipped     |2    |
|100145  |C0007      |Sai Mishra      |Face Wash        |Beauty         |7       |350       |30              |Cash on Delivery|Bengaluru |Karnataka     |2025-08-28|Delivered   |2    |
|100297  |C0149      |Manish Pillai   |Dry Fruits P

In [14]:
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

null_counts.show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [15]:
string_columns = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
]

print("String columns:")
print(string_columns)

String columns:
['customer_id', 'customer_name', 'product', 'category', 'payment_method', 'city', 'state', 'order_date', 'order_status']


In [16]:

df.select("customer_name").distinct().show(100, truncate=False)
df.select("product").distinct().show(100, truncate=False)
df.select("category").distinct().show(100, truncate=False)
df.select("payment_method").distinct().show(100, truncate=False)
df.select("city").distinct().show(100, truncate=False)
df.select("state").distinct().show(100, truncate=False)
df.select("order_status").distinct().show(100, truncate=False)

+------------------+
|customer_name     |
+------------------+
|Pranav Jain       |
|Tanvi Rao         |
|Meera Pandey      |
|Tanvi Patel       |
|Rahul Jain        |
|Kavya Bose        |
|Siddharth Menon   |
|Divya Naidu       |
|Aditi Agarwal     |
|Aman Naidu        |
|Siddharth Malhotra|
|Sneha Chopra      |
|Vikram Reddy      |
|Harsh Iyer        |
|Tanvi Naidu       |
|Swathi Gupta      |
|Shreya Gupta      |
|Harsh Pillai      |
|Pooja Jain        |
|Siddharth Kapoor  |
|Keerthi Tripathi  |
|Aditya Chopra     |
|Arjun Agarwal     |
|Vivaan Nair       |
|Deepak Pandey     |
|Pooja Verma       |
|Neha Sharma       |
|Divya Chopra      |
|Aarav Nair        |
|Kavya Malhotra    |
|Sanjay Shetty     |
|Vivaan Pandey     |
|Aditi Patel       |
|Nandini Das       |
|Aditi Tripathi    |
|Siddharth Nair    |
|Divya Sharma      |
|Ananya Verma      |
|Ananya Mishra     |
|Meera Reddy       |
|Varun Agarwal     |
|Pranav Agarwal    |
|Siddharth Iyer    |
|Kavya Sharma      |
|Sakshi Tripa

In [17]:
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [18]:
# For cols : quantity, unit_price, discount_percent
df.select(
    F.min("quantity").alias("min_quantity"),
    F.max("quantity").alias("max_quantity"),
    F.avg("quantity").alias("avg_quantity")
).show()

df.select(
    F.min("unit_price").alias("min_unit_price"),
    F.max("unit_price").alias("max_unit_price"),
    F.avg("unit_price").alias("avg_unit_price")
).show()

df.select(
    F.min("unit_price").alias("min_unit_price"),
    F.max("unit_price").alias("max_unit_price"),
    F.avg("unit_price").alias("avg_unit_price")
).show()

+------------+------------+------------+
|min_quantity|max_quantity|avg_quantity|
+------------+------------+------------+
|          -2|          10|       5.355|
+------------+------------+------------+

+--------------+--------------+--------------+
|min_unit_price|max_unit_price|avg_unit_price|
+--------------+--------------+--------------+
|          -500|         81010|        5702.4|
+--------------+--------------+--------------+

+--------------+--------------+--------------+
|min_unit_price|max_unit_price|avg_unit_price|
+--------------+--------------+--------------+
|          -500|         81010|        5702.4|
+--------------+--------------+--------------+



In [19]:
# quantity

df.filter(
    (F.col("quantity") <= 0) | F.col("quantity").isNull()
).show(100, truncate=False)

print(
    "Invalid quantity records:",
    df.filter(
        (F.col("quantity") <= 0) | F.col("quantity").isNull()
    ).count()
)

+--------+-----------+-------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name|product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+-------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100114  |C0252      |Keerthi Singh|Ceiling Fan           |Home Appliances|0       |2920      |15              |Net Banking     |Chennai   |Tamil Nadu    |2025-02-15|Delivered   |
|100143  |C0184      |Aman Joshi   |Football              |Sports         |0       |1030      |15              |Debit Card      |Coimbatore|Tamil Nadu    |2025-11-26|Delivered   |
|100198  |C0014      |Arjun Pillai |Hoodie                |Clothing       |-2      |2160      |12   

In [20]:
# unit_price

df.filter(
    (F.col("unit_price") <= 0) | F.col("unit_price").isNull()
).show(100, truncate=False)

print(
    "Invalid unit price records:",
    df.filter(
        (F.col("unit_price") <= 0) | F.col("unit_price").isNull()
    ).count()
)

+--------+-----------+----------------+----------------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name   |product               |category   |quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+----------------+----------------------+-----------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|100102  |C0032      |Anjali Kumar    |Atta 10kg             |Grocery    |7       |0         |40              |Credit Card   |Mumbai    |Maharashtra   |2025-03-02|Processing  |
|100170  |C0158      |Deepak Pandey   |Competitive Exam Guide|Books      |6       |0         |0               |Net Banking   |Pune      |Maharashtra   |2025-08-26|Shipped     |
|100235  |C0349      |Tanvi Reddy     |Power Bank            |Electronics|7       |-500      |0               |UPI 

In [21]:
# discount_percent

df.filter(
    (F.col("discount_percent") < 0) |
    (F.col("discount_percent") > 100) |
    F.col("discount_percent").isNull()
).show(100, truncate=False)

print(
    "Invalid discount records:",
    df.filter(
        (F.col("discount_percent") < 0) |
        (F.col("discount_percent") > 100) |
        F.col("discount_percent").isNull()
    ).count()
)

+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+
|order_id|customer_id|customer_name  |product         |category       |quantity|unit_price|discount_percent|payment_method  |city      |state     |order_date|order_status|
+--------+-----------+---------------+----------------+---------------+--------+----------+----------------+----------------+----------+----------+----------+------------+
|100034  |C0192      |Divya Sharma   |Yoga Mat        |Sports         |7       |1720      |-5              |Cash on Delivery|Coimbatore|Tamil Nadu|2025-08-19|Delivered   |
|100140  |C0075      |Swathi Tripathi|Cooking Oil 5L  |Grocery        |4       |790       |-5              |Credit Card     |Hyderabad |Telangana |2025-10-01|Shipped     |
|100217  |C0214      |Neha Tripathi  |Football        |Sports         |1       |650       |-5              |UPI             |Chennai   |Tami

In [23]:
# Silver DataFrame
silver_df = df

In [24]:
silver_df = silver_df.dropDuplicates()

print(
    "Rows after removing duplicates:",
    silver_df.count()
)

Rows after removing duplicates: 990


In [25]:
text_columns = [
    "customer_id",
    "customer_name",
    "product",
    "category",
    "payment_method",
    "city",
    "state",
    "order_status"
]

for col_name in text_columns:
    silver_df = silver_df.withColumn(
        col_name,
        F.trim(F.col(col_name))
    )

In [26]:
capitalize_columns = [
    "customer_name",
    "product",
    "category",
    "city",
    "state",
    "order_status"
]

for col_name in capitalize_columns:
    silver_df = silver_df.withColumn(
        col_name,
        F.initcap(F.col(col_name))
    )

In [27]:
silver_df.select("payment_method").distinct().show(100, truncate=False)

+----------------+
|payment_method  |
+----------------+
|Credit Card     |
|Net Banking     |
|Debit Card      |
|Cash on Delivery|
|UPI             |
|NULL            |
+----------------+



In [28]:
silver_df.filter(F.col("payment_method").isNull()).count()

8

In [29]:
silver_df = silver_df.fillna({
    "payment_method": "Unknown"
})

In [30]:
silver_df.select("payment_method").distinct().show(100, truncate=False)

+----------------+
|payment_method  |
+----------------+
|Credit Card     |
|Unknown         |
|Net Banking     |
|Debit Card      |
|Cash on Delivery|
|UPI             |
+----------------+



In [31]:
silver_df = silver_df.withColumn(
    "quantity",
    F.when(
        F.col("quantity") > 0,
        F.col("quantity")
    ).otherwise(None)
)

In [32]:
quantity_median = silver_df.approxQuantile(
    "quantity",
    [0.5],
    0.01
)[0]

print("Quantity median:", quantity_median)

Quantity median: 5.0


In [33]:
silver_df = silver_df.fillna({
    "quantity": int(quantity_median)
})

In [34]:
silver_df = silver_df.withColumn(
    "unit_price",
    F.when(
        F.col("unit_price") > 0,
        F.col("unit_price")
    ).otherwise(None)
)

In [35]:
unit_price_median = silver_df.approxQuantile(
    "unit_price",
    [0.5],
    0.01
)[0]

print("Unit price median:", unit_price_median)

Unit price median: 1810.0


In [36]:
silver_df = silver_df.fillna({
    "unit_price": int(unit_price_median)
})

In [37]:
silver_df = silver_df.withColumn(
    "discount_percent",
    F.when(
        (F.col("discount_percent") >= 0) &
        (F.col("discount_percent") <= 100),
        F.col("discount_percent")
    ).otherwise(None)
)

In [38]:
discount_median = silver_df.approxQuantile(
    "discount_percent",
    [0.5],
    0.01
)[0]

print("Discount median:", discount_median)

Discount median: 20.0


In [39]:
silver_df = silver_df.fillna({
    "discount_percent": int(discount_median)
})

In [40]:
silver_df = silver_df.fillna({
    "customer_name": "Unknown",
    "product": "Unknown",
    "category": "Unknown",
    "payment_method": "Unknown",
    "city": "Unknown",
    "state": "Unknown",
    "order_status": "Unknown"
})

In [41]:
silver_df.select("order_date") \
    .distinct() \
    .show(100, truncate=False)

+----------+
|order_date|
+----------+
|2026-02-01|
|2025-06-03|
|2025-02-25|
|2026-02-27|
|2025-08-17|
|2026-07-23|
|2026-07-04|
|2025-09-14|
|2025-03-20|
|2025-04-05|
|2026-04-15|
|2026-06-05|
|2025-01-03|
|2025-12-07|
|2026-03-15|
|2025-12-30|
|2025-08-18|
|2026-09-11|
|2025-06-16|
|2026-09-06|
|2025-07-15|
|2026-08-24|
|2025/07/14|
|2025-05-06|
|2025-04-12|
|2026-02-15|
|2025-07-21|
|2025-05-23|
|2025-10-09|
|2026-01-05|
|2025-01-23|
|2026-01-28|
|2025-06-14|
|2026-02-25|
|2025-07-16|
|2025-01-28|
|2026-01-09|
|2025-03-12|
|2025-03-26|
|2025-12-14|
|2025-07-06|
|2025-07-08|
|2026-01-02|
|2025-05-09|
|2025-03-07|
|2025-06-01|
|2025-06-30|
|2025-07-29|
|2025-01-22|
|2026-01-13|
|2026-01-08|
|2026-03-10|
|2026-07-10|
|2026-04-24|
|2025-10-26|
|2025-02-15|
|2026-06-22|
|2025-02-05|
|2025-04-20|
|2025-11-11|
|31/02/2026|
|2025-04-10|
|2026-01-22|
|2025-07-09|
|2025-07-07|
|2026-01-19|
|2026-01-31|
|2025-10-21|
|2025-02-04|
|2025-04-04|
|2026-05-19|
|2026-03-21|
|2025-09-27|
|2026-02-28|

In [42]:
silver_df.select(
    "order_id",
    "order_date"
).show(50, truncate=False)

+--------+----------+
|order_id|order_date|
+--------+----------+
|100331  |2025-06-06|
|100333  |2025-05-08|
|100506  |2025-06-02|
|100931  |2025-12-25|
|100168  |2026-06-04|
|100473  |2026-03-24|
|100532  |2026-04-30|
|100011  |2026-07-27|
|100064  |2025-05-15|
|100176  |2026-08-23|
|100307  |2025-12-25|
|100414  |2025-05-06|
|100680  |2026-05-05|
|100018  |2026-08-17|
|100060  |2026-01-18|
|100275  |2025-03-05|
|100332  |2025-12-06|
|100667  |2026-03-04|
|100865  |2026-02-01|
|100883  |2026-08-19|
|100101  |2026-09-12|
|100375  |2025-05-03|
|100525  |2025-12-29|
|100919  |2025-02-06|
|100928  |2025-05-19|
|100982  |2026-06-02|
|100130  |2026-08-03|
|100202  |2026-02-12|
|100610  |2026-03-16|
|100619  |2026-01-09|
|100793  |2026-03-27|
|100829  |2025-05-01|
|100849  |2025-01-30|
|100907  |2026-01-23|
|100207  |2026-01-28|
|100585  |2026-05-27|
|100623  |2025-10-31|
|100010  |2026-08-24|
|100589  |2026-05-31|
|100612  |2025-11-20|
|100628  |2026-05-04|
|100887  |2025-05-02|
|100252  |

In [47]:
silver_df = silver_df.withColumn(
    "parsed_order_date",
    F.coalesce(
        F.to_date(
            F.try_to_timestamp(F.col("order_date"), F.lit("yyyy-MM-dd"))
        ),
        F.to_date(
            F.try_to_timestamp(F.col("order_date"), F.lit("dd/MM/yyyy"))
        ),
        F.to_date(
            F.try_to_timestamp(F.col("order_date"), F.lit("MM/dd/yyyy"))
        ),
        F.to_date(
            F.try_to_timestamp(F.col("order_date"), F.lit("yyyy/MM/dd"))
        ),
        F.to_date(
            F.try_to_timestamp(F.col("order_date"), F.lit("dd-MM-yyyy"))
        ),
        F.to_date(
            F.try_to_timestamp(F.col("order_date"), F.lit("MM-dd-yyyy"))
        )
    )
)

In [44]:
silver_df.select(
    "order_date",
    "parsed_order_date"
).show(100, truncate=False)

+----------+-----------------+
|order_date|parsed_order_date|
+----------+-----------------+
|2025-06-06|2025-06-06       |
|2025-05-08|2025-05-08       |
|2025-06-02|2025-06-02       |
|2025-12-25|2025-12-25       |
|2026-06-04|2026-06-04       |
|2026-03-24|2026-03-24       |
|2026-04-30|2026-04-30       |
|2026-07-27|2026-07-27       |
|2025-05-15|2025-05-15       |
|2026-08-23|2026-08-23       |
|2025-12-25|2025-12-25       |
|2025-05-06|2025-05-06       |
|2026-05-05|2026-05-05       |
|2026-08-17|2026-08-17       |
|2026-01-18|2026-01-18       |
|2025-03-05|2025-03-05       |
|2025-12-06|2025-12-06       |
|2026-03-04|2026-03-04       |
|2026-02-01|2026-02-01       |
|2026-08-19|2026-08-19       |
|2026-09-12|2026-09-12       |
|2025-05-03|2025-05-03       |
|2025-12-29|2025-12-29       |
|2025-02-06|2025-02-06       |
|2025-05-19|2025-05-19       |
|2026-06-02|2026-06-02       |
|2026-08-03|2026-08-03       |
|2026-02-12|2026-02-12       |
|2026-03-16|2026-03-16       |
|2026-01

In [48]:
invalid_dates = silver_df.filter(
    F.col("parsed_order_date").isNull() &
    F.col("order_date").isNotNull()
)

print(
    "Invalid/unparseable date records:",
    invalid_dates.count()
)

invalid_dates.select(
    "order_id",
    "order_date"
).show(100, truncate=False)

Invalid/unparseable date records: 4
+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |31/02/2026|
|100637  |2026-13-05|
|100448  |not-a-date|
|100634  |2025-02-30|
+--------+----------+



In [49]:
silver_df = silver_df.drop("order_date")

silver_df = silver_df.withColumnRenamed(
    "parsed_order_date",
    "order_date"
)

In [50]:
silver_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- unit_price: integer (nullable = false)
 |-- discount_percent: integer (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_status: string (nullable = false)
 |-- order_date: date (nullable = true)



In [52]:
silver_df.show(20, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+------------+----------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_status|order_date|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+------------+----------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |Cancelled   |2025-06-06|
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |Delivered   |2025-05-08|
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books      

In [53]:
silver_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: integer (nullable = false)
 |-- unit_price: integer (nullable = false)
 |-- discount_percent: integer (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_status: string (nullable = false)
 |-- order_date: date (nullable = true)



In [54]:
print("Current Silver row count:", silver_df.count())

Current Silver row count: 990


In [55]:
silver_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in silver_df.columns
]).show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+------------+----------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_status|order_date|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+------------+----------+
|       0|          0|            0|      0|       0|       0|         0|               0|             0|   0|    0|           0|         4|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+------------+----------+



In [56]:
silver_df.filter(
    F.col("order_date").isNull()
).select(
    "order_id",
    "order_date"
).show(20, truncate=False)

+--------+----------+
|order_id|order_date|
+--------+----------+
|100097  |NULL      |
|100637  |NULL      |
|100448  |NULL      |
|100634  |NULL      |
+--------+----------+



In [57]:
silver_path = "data_lake/silver/preprocessed_sales"

silver_df.write \
    .mode("overwrite") \
    .parquet(silver_path)

print("Silver data saved successfully.")

Silver data saved successfully.


In [58]:
import os

for root, dirs, files in os.walk("data_lake"):
    level = root.replace("data_lake", "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

data_lake/
    bronze/
        sales.csv
    silver/
        preprocessed_sales/
            .part-00000-9e8303c3-882f-4f5a-86f6-b7de60fa247d-c000.snappy.parquet.crc
            part-00000-9e8303c3-882f-4f5a-86f6-b7de60fa247d-c000.snappy.parquet
            ._SUCCESS.crc
            _SUCCESS


In [59]:
silver_read = spark.read.parquet(
    "data_lake/silver/preprocessed_sales"
)

print("Silver data read successfully.")

Silver data read successfully.


In [60]:
silver_read.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_date: date (nullable = true)



In [61]:
silver_read.show(10, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+------------+----------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_status|order_date|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+------------+----------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka     |Cancelled   |2025-06-06|
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra   |Delivered   |2025-05-08|
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books      

In [62]:
after_count = silver_read.count()

print("Rows before preprocessing:", before_count)
print("Rows after preprocessing :", after_count)
print("Rows removed             :", before_count - after_count)

Rows before preprocessing: 1000
Rows after preprocessing : 990
Rows removed             : 10


In [63]:
silver_read.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in silver_read.columns
]).show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+------------+----------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_status|order_date|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+------------+----------+
|       0|          0|            0|      0|       0|       0|         0|               0|             0|   0|    0|           0|         4|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+------------+----------+



In [64]:
print("=" * 60)
print("FINAL VERIFICATION")
print("=" * 60)

print("\nBronze Layer:")
print("data_lake/bronze/sales.csv")

print("\nSilver Layer:")
print("data_lake/silver/preprocessed_sales")

print("\nRow Counts:")
print("Before preprocessing:", before_count)
print("After preprocessing :", after_count)
print("Rows removed        :", before_count - after_count)

print("\nSilver Schema:")
silver_read.printSchema()

print("\nSample Silver Records:")
silver_read.show(5, truncate=False)

FINAL VERIFICATION

Bronze Layer:
data_lake/bronze/sales.csv

Silver Layer:
data_lake/silver/preprocessed_sales

Row Counts:
Before preprocessing: 1000
After preprocessing : 990
Rows removed        : 10

Silver Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_date: date (nullable = true)


Sample Silver Records:
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+------------+----------+
|order_id|customer_id|customer_

## Preprocessing Operations Performed

The raw sales.csv file was preserved unchanged in the Bronze layer.

The following preprocessing operations were performed using Apache Spark:

1. Identified and removed exact duplicate records.
2. Checked for missing values in all columns.
3. Trimmed leading and trailing spaces from text fields.
4. Standardized obvious capitalization inconsistencies in applicable text fields.
5. Inspected payment method values and handled missing payment methods using "Unknown".
6. Validated quantity values and treated non-positive values as invalid.
7. Validated unit price values and treated non-positive values as invalid.
8. Validated discount percentage values to ensure they were between 0 and 100.
9. Handled missing and invalid numeric values using median-based imputation.
10. Converted valid order dates to a consistent date format.
11. Identified 4 unparseable/invalid order dates and retained those records with NULL dates rather than deleting the records.
12. Saved the processed data in Parquet format in the Silver layer.
13. Read the Silver data back using Spark and verified the schema, sample records, and row counts.